some adjustments to be made
0. Make sure all the dimensions follow the main logic of the matrix
1. Add the parameters for the tensor dimensions for the attention block.
2. Add the masking matrix to the self-attention block
3. Double check that all the multiplications go as planned. I recieve the expected outcomes.

In [1]:
import math
import torch
import torch.nn as nn

In [11]:
D = 4 # number of dimensions (for the embeddings as far as i am concerned)
I = 5 # number of input tokens (as far as i am concerned
N_ATT = 1

In [12]:
X = torch.rand((D, I)) # Dimensions are rows, and I-inputs are individual tokens

In [13]:
class SelfAttention_Block(nn.Module):    
    def __init__(self, num_att_heads: int, D_dimension: int):
        super().__init__()
        self.d_k = D_dimension // num_att_heads            # head size
        # linear projections  W_*  ∈ ℝ[d_k × D]
        self.Q_w = nn.Parameter(torch.randn(self.d_k, D_dimension))
        self.K_w = nn.Parameter(torch.randn(self.d_k, D_dimension))
        self.V_w = nn.Parameter(torch.randn(self.d_k, D_dimension))

        self.Q_b = nn.Parameter(torch.randn(self.d_k, 1))
        self.K_b = nn.Parameter(torch.randn(self.d_k, 1))
        self.V_b = nn.Parameter(torch.randn(self.d_k, 1))

        self.softmax = nn.Softmax(dim=1) 

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # 1) linear projections
        Q = self.Q_w @ X + self.Q_b            # (d_k, N)
        K = self.K_w @ X + self.K_b
        V = self.V_w @ X + self.V_b

        # 2) scaled dot-product attention
        attn = (Q.t() @ K) / math.sqrt(self.d_k)         # (N, N)
        attn   = self.softmax(attn)                      # (N, N)

        # 3) weighted sum of values – note Vᵀ to get (N, d_k)
        context = (attn @ V.t()).t()                       # (d_k, N)
        return context

In [14]:
block = SelfAttention_Block(1, D)
out   = block(X)           # forward

loss  = out.sum()          # scalar, any criterion ok
block.zero_grad()          # clear
loss.backward()            # backward pass

print(block.K_w.grad)   # leaf param grad
print(block.V_w.grad)     # non-leaf grad (only if you saved it)

tensor([[ 0.0155,  0.0158, -0.0255,  0.0225],
        [-0.1209, -0.1487,  0.1731, -0.1718],
        [ 0.1125,  0.1216, -0.1852,  0.1705],
        [-0.1664, -0.2056,  0.2419, -0.2350]])
tensor([[3.4275, 1.8247, 2.9026, 1.1715],
        [3.4275, 1.8247, 2.9026, 1.1715],
        [3.4275, 1.8247, 2.9026, 1.1715],
        [3.4275, 1.8247, 2.9026, 1.1715]])
